In [7]:
import cv2
import dlib
import numpy as np
import os

In [8]:
predictor_path = 'shape_predictor_68_face_landmarks.dat'
face_rec_model_path = 'dlib_face_recognition_resnet_model_v1.dat'

detector = dlib.get_frontal_face_detector()
sp = dlib.shape_predictor(predictor_path)
facerec = dlib.face_recognition_model_v1(face_rec_model_path)

In [9]:
def cargar_rostros_y_embeddings(carpeta_rostros='rostros'):
    nombres = []
    descriptores = []

    for archivo in os.listdir(carpeta_rostros):
        if archivo.endswith('.jpg') or archivo.endswith('.png'):
            img_path = os.path.join(carpeta_rostros, archivo)
            img = dlib.load_rgb_image(img_path)

            dets = detector(img, 1)
            if len(dets) > 0:
                shape = sp(img, dets[0])
                face_descriptor = facerec.compute_face_descriptor(img, shape)
                descriptores.append(np.array(face_descriptor))
                nombre = os.path.splitext(archivo)[0]
                nombres.append(nombre)
            else:
                print(f"No se detectó rostro en {archivo}")
    return nombres, descriptores

nombres_registrados, descriptores_registrados = cargar_rostros_y_embeddings()

In [10]:
def reconocer_rostro(frame):
    rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    dets = detector(rgb, 1)

    for d in dets:
        shape = sp(rgb, d)
        face_descriptor = facerec.compute_face_descriptor(rgb, shape)
        descriptor_actual = np.array(face_descriptor)

        # Comparar con cada rostro registrado
        distancias = [np.linalg.norm(descriptor_actual - desc) for desc in descriptores_registrados]
        idx = np.argmin(distancias)

        if distancias[idx] < 0.6:
            nombre = nombres_registrados[idx]
        else:
            nombre = "Desconocido"

        # Dibujar recuadro y nombre
        x1, y1, x2, y2 = d.left(), d.top(), d.right(), d.bottom()
        cv2.rectangle(frame, (x1, y1), (x2, y2), (0,255,0), 2)
        cv2.putText(frame, nombre, (x1, y1-10), cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0,255,0), 2)

    return frame

In [11]:
cap = cv2.VideoCapture(0)

while True:
    ret, frame = cap.read()
    if not ret:
        break

    frame_procesado = reconocer_rostro(frame)
    cv2.imshow("Reconocimiento Facial", frame_procesado)

    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()